# Baseline Model Training

Trains a binary classification model on stable synthetic data.
Saves three artifacts:
- `artifacts/baseline_model.joblib` — trained model
- `artifacts/reference_data.csv` — training dataset (Evidently reference)
- `artifacts/model_metadata.json` — version info + performance metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

from core.config import get_settings
from data.simulator import generate_production_batch
from datetime import date

settings = get_settings()
settings.artifacts_dir.mkdir(exist_ok=True)

In [ ]:
# Generate stable training data (1000 samples)
frames = [
    generate_production_batch('stable', date(2024, m, d), n_samples=100, random_state=m * 31 + d)
    for m in range(1, 11)
    for d in [1, 15]
]
df = pd.concat(frames, ignore_index=True)
df = df.drop(columns=['batch_date'])
print(f'Training dataset: {df.shape}')
df.describe()

In [ ]:
feature_cols = settings.feature_names
target_col = settings.target_column

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Class balance (train): {y_train.value_counts().to_dict()}')

In [ ]:
model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
print('Model trained.')

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy':  round(accuracy_score(y_test, y_pred), 4),
    'precision': round(precision_score(y_test, y_pred), 4),
    'recall':    round(recall_score(y_test, y_pred), 4),
    'f1':        round(f1_score(y_test, y_pred), 4),
    'roc_auc':   round(roc_auc_score(y_test, y_proba), 4),
}
print('Reference metrics:', metrics)

In [ ]:
# Save artifacts
joblib.dump(model, settings.model_path)
print(f'Model saved to {settings.model_path}')

df.to_csv(settings.reference_data_path, index=False)
print(f'Reference data saved to {settings.reference_data_path}')

metadata = {
    'version': '1.0.0',
    'model_type': 'GradientBoostingClassifier',
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'n_train_samples': int(X_train.shape[0]),
    'n_test_samples': int(X_test.shape[0]),
    'features': feature_cols,
    'target': target_col,
    'reference_metrics': metrics,
    'model_path': str(settings.model_path),
}
settings.model_metadata_path.write_text(json.dumps(metadata, indent=2))
print(f'Metadata saved to {settings.model_metadata_path}')

In [ ]:
# Verify reload
loaded = joblib.load(settings.model_path)
proba_check = loaded.predict_proba(X_test[:5])[:, 1]
assert all(0 <= p <= 1 for p in proba_check), 'Probabilities out of range'
print('Reload check passed. Probabilities:', proba_check.round(3))